In [2]:
import numpy as np
import pandas as pd

import once, this shit is huge - 5g

In [ ]:


train = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/train.csv',  )




/var/folders/5p/tsf09yfn1d7ct362cxy57hyc0000gn/T/ipykernel_19565/2690657249.py:1: DtypeWarning: Columns (0: onpromotion) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/train.csv')


In [ ]:
print(train.shape)
print(train.head())
train['onpromotion'] # onpromotion is boolean but also has NaN values, well need to decide how to clean it.
promotion = train['onpromotion'].copy().drop_duplicates()
# train

(125497040, 6)
   id        date  store_nbr  item_nbr  unit_sales onpromotion
0   0  2013-01-01         25    103665         7.0         NaN
1   1  2013-01-01         25    105574         1.0         NaN
2   2  2013-01-01         25    105575         2.0         NaN
3   3  2013-01-01         25    108079         1.0         NaN
4   4  2013-01-01         25    108701         1.0         NaN


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN
...,...,...,...,...,...,...
125497035,125497035,2017-08-15,54,2089339,4.0,False
125497036,125497036,2017-08-15,54,2106464,1.0,True
125497037,125497037,2017-08-15,54,2110456,192.0,False
125497038,125497038,2017-08-15,54,2113914,198.0,True


In [ ]:
total_daily_sales = train.groupby('store_nbr')

In [11]:
item_nbr = train['item_nbr'].copy()
item_nbr.drop_duplicates()


0             103665
1             105574
2             105575
3             108079
4             108701
              ...   
125121287    2122818
125163650    2011459
125163892    2126944
125193855    2123839
125309124    2011451
Name: item_nbr, Length: 4036, dtype: int64

apparently 4036 unique items

notes for holidays:

Wages in the public sector are paid every two weeks on the 15 th and on the last day of the month. Supermarket sales could be affected by this.
A magnitude 7.8 earthquake struck Ecuador on April 16, 2016. People rallied in relief efforts donating water and other first need products which greatly affected supermarket sales for several weeks after the earthquake.

In [ ]:
holi = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/holidays_events.csv')
print(holi.shape)
print(holi.head())
holi
clean_holi = holi.drop(['description', 'transferred'], axis=1)
# clean_holi.merge()
# clean_holi
# holi['type'].unique() 
holi['transferred'].

(350, 6)
         date     type    locale locale_name                    description  \
0  2012-03-02  Holiday     Local       Manta             Fundacion de Manta   
1  2012-04-01  Holiday  Regional    Cotopaxi  Provincializacion de Cotopaxi   
2  2012-04-12  Holiday     Local      Cuenca            Fundacion de Cuenca   
3  2012-04-14  Holiday     Local    Libertad      Cantonizacion de Libertad   
4  2012-04-21  Holiday     Local    Riobamba      Cantonizacion de Riobamba   

   transferred  
0        False  
1        False  
2        False  
3        False  
4        False  


<StringArray>
['Holiday', 'Transfer', 'Additional', 'Bridge', 'Work Day', 'Event']
Length: 6, dtype: str

In [13]:
items = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/items.csv')
print(items.shape)
print(items.head())

(4100, 4)
   item_nbr        family  class  perishable
0     96995     GROCERY I   1093           0
1     99197     GROCERY I   1067           0
2    103501      CLEANING   3008           0
3    103520     GROCERY I   1028           0
4    103665  BREAD/BAKERY   2712           1


In [ ]:
item_nbr

In [14]:
oil = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/oil.csv')
print(oil.shape)
print(oil.head())

(1218, 2)
         date  dcoilwtico
0  2013-01-01         NaN
1  2013-01-02       93.14
2  2013-01-03       92.97
3  2013-01-04       93.12
4  2013-01-07       93.20


In [48]:
stores = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/stores.csv')
print(stores.shape)
print(stores.head())
clean_stores = stores.drop(['type', 'cluster'], axis=1)
clean_stores


(54, 5)
   store_nbr           city                           state type  cluster
0          1          Quito                       Pichincha    D       13
1          2          Quito                       Pichincha    D       13
2          3          Quito                       Pichincha    D        8
3          4          Quito                       Pichincha    D        9
4          5  Santo Domingo  Santo Domingo de los Tsachilas    D        4


,store_nbr,city,state
0,1,Quito,Pichincha
1,2,Quito,Pichincha
2,3,Quito,Pichincha
3,4,Quito,Pichincha
4,5,Santo Domingo,Santo Domingo de los Tsachilas
5,6,Quito,Pichincha
6,7,Quito,Pichincha
7,8,Quito,Pichincha
8,9,Quito,Pichincha
9,10,Quito,Pichincha


In [3]:
transactions = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/transactions.csv')
print(transactions.shape)
print(transactions.head())

(83488, 3)
         date  store_nbr  transactions
0  2013-01-01         25           770
1  2013-01-02          1          2111
2  2013-01-02          2          2358
3  2013-01-02          3          3487
4  2013-01-02          4          1922


In [70]:
daily_trans_per_store = transactions.merge(right=clean_stores, on='store_nbr')
# holidays= daily_trans_per_store.copy().merge(right=holi, on=['city', 'state', 'locale_name'])
daily_trans_per_store

# daily_trans_total = transactions.groupby('store_nbr')
# daily_trans_total

ts = daily_trans_per_store.pivot(index='date', columns='store_nbr', values='transactions')
# ts.fillna(0)
filledts= ts.fillna(0)
# ts
filledts
# filledts.to_csv('../data/cleaned/daily_store_transactions')


store_nbr,1,2,3,4,5,6,7,8,9,10,...,45,46,47,48,49,50,51,52,53,54
date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2013-01-02,2111.0,2358.0,3487.0,1922.0,1903.0,2143.0,1874.0,3250.0,2940.0,1293.0,...,4208.0,4886.0,4161.0,3397.0,2346.0,3077.0,1985.0,0.0,0.0,998.0
2013-01-03,1833.0,2033.0,3026.0,1551.0,1740.0,1795.0,1568.0,2904.0,2396.0,1157.0,...,3314.0,3438.0,3660.0,2887.0,1702.0,2307.0,1644.0,0.0,0.0,920.0
2013-01-04,1863.0,2066.0,3188.0,1596.0,1642.0,1679.0,1513.0,2962.0,1975.0,970.0,...,3630.0,3434.0,3915.0,2900.0,2016.0,2698.0,1786.0,0.0,0.0,794.0
2013-01-05,1509.0,2062.0,3623.0,1825.0,1643.0,2154.0,1599.0,3060.0,2604.0,1269.0,...,4331.0,4935.0,4764.0,4084.0,2562.0,3459.0,2068.0,0.0,0.0,949.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2017-08-11,570.0,1698.0,2991.0,1301.0,1183.0,1747.0,1541.0,2212.0,1848.0,728.0,...,4302.0,3814.0,4009.0,3220.0,3117.0,2971.0,1922.0,2957.0,1272.0,768.0
2017-08-12,1004.0,1613.0,3070.0,1304.0,1061.0,1706.0,1612.0,2463.0,1920.0,953.0,...,3994.0,3697.0,3825.0,3198.0,2985.0,2987.0,1590.0,2804.0,1212.0,903.0
2017-08-13,416.0,1658.0,3075.0,1378.0,1098.0,1781.0,1410.0,2355.0,1745.0,810.0,...,4054.0,3839.0,3741.0,3381.0,3028.0,2826.0,1816.0,2433.0,1164.0,1054.0


In [ ]:
sample_submission = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/sample_submission.csv')
print(sample_submission.shape)
print(sample_submission.head())

In [ ]:
test = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/test.csv')
print(test.shape)
print(test.head())

some ideas from the data that make sense to me to join: 

item_nbr + family 